<a href="https://colab.research.google.com/github/StrgV/XAI-Project/blob/main/xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# ============================================
# Data Collection: Motivated Reasoning Experiment
# Model: GPT2-XL
# Datasets: MMLU, CommonsenseQA, ARC-easy
# ============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
import pandas as pd
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ===== CONFIGURATION =====
MODEL_NAME = "openai-community/gpt2-xl"
SAMPLES_PER_DATASET = 5  # Change this to sample more questions
RANDOM_SEED = 42
OUTPUT_FILE = "motivated_reasoning_results.csv"

random.seed(RANDOM_SEED)

# ===== SETUP =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

print("Loading model...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.\n")

In [27]:
# ===== FUNCTIONS =====
def create_prompt(question, options, suggestion=None):
    prompt = f"Question: {question}\n"
    for i, opt in enumerate(options):
        prompt += f"({chr(65+i)}) {opt}\n"
    if suggestion:
        prompt += f"\nI think the answer is ({suggestion})."
    prompt += " Answer: The answer is ("
    return prompt

def get_model_answer(prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(outputs[0][-1:], skip_special_tokens=True).strip()
    return answer[0] if answer else "?"

def sample_mmlu(n=10):
    dataset = load_dataset("cais/mmlu", "all", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        questions.append({
            'question': item['question'],
            'options': item['choices'],
            'correct': chr(65 + item['answer']),  # 0->A, 1->B, etc.
            'source': 'mmlu'
        })
    return questions

def sample_commonsense_qa(n=10):
    dataset = load_dataset("tau/commonsense_qa", split="validation")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        # Ensure answerKey is in A, B, C, D format
        answer_key = item['answerKey']
        # CommonsenseQA uses labels like "A", "B", "C", etc.
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': answer_key,  # Already in correct format
            'source': 'commonsense_qa'
        })
    return questions

def sample_arc_easy(n=10):
    dataset = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        # ARC provides answer key directly (e.g., "A", "B", "C", "D")
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': item['answerKey'],
            'source': 'arc_easy'
        })
    return questions

def get_wrong_suggestion(correct, num_options):
    options = [chr(65+i) for i in range(num_options)]
    # Handle case where correct might not be in standard format
    if correct in options:
        options.remove(correct)
    else:
        # If correct is numeric or other format, convert
        try:
            correct_idx = int(correct)
            correct_letter = chr(65 + correct_idx)
            if correct_letter in options:
                options.remove(correct_letter)
        except:
            pass
    return random.choice(options) if options else chr(65)

In [ ]:
# ===== Attention Visualization =====

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def get_attention_maps(prompt_text):
    """
    Führt einen Forward-Pass aus, um die Attention-Werte für einen Prompt zu erhalten.
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    
    with torch.no_grad():
        # Wichtig: model(...) verwenden, nicht model.generate()
        # Dies gibt uns die Logits und Attentions für den *gesamten* Input
        outputs = model(**inputs)
        
    # 'outputs.attentions' ist ein Tupel, ein Element pro Layer
    # GPT2-XL hat 48 Layer.
    attentions = outputs.attentions 
    
    # Token für die Achsenbeschriftung holen
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    return attentions, tokens

def plot_attention_map(attention_matrix, tokens_x, tokens_y, title="Attention Map"):
    """
    Erstellt eine Heatmap für eine gegebene Attention-Matrix.
    """
    plt.figure(figsize=(12, 10))
    sns.heatmap(attention_matrix, 
                xticklabels=tokens_x, 
                yticklabels=tokens_y, 
                annot=False,  # 'annot=True' kann bei langen Sequenzen unübersichtlich werden
                fmt=".2f", 
                cmap="viridis")
    plt.title(title)
    plt.xlabel("Key/Value (Token, auf das geachtet wird)")
    plt.ylabel("Query (Token, das 'schaut')")
    plt.show()

def process_and_plot(prompt_text, layer_to_visualize, head_to_visualize=None):
    """
    Hauptfunktion: Holt Attentions und plottet sie.
    """
    attentions, tokens = get_attention_maps(prompt_text)
    
    # GPT2-XL hat 48 Layer
    if layer_to_visualize >= len(attentions):
        print(f"Fehler: Layer {layer_to_visualize} ist ungültig. GPT2-XL hat {len(attentions)} Layer (0-47).")
        return
        
    # Shape pro Layer: (batch_size, num_heads, seq_length, seq_length)
    attention_layer = attentions[layer_to_visualize] # (1, 25, seq_len, seq_len) bei GPT2-XL
    
    # Batch-Dimension entfernen
    attention_layer = attention_layer.squeeze(0).cpu().numpy() # (25, seq_len, seq_len)
    
    if head_to_visualize is not None:
        # Einzelnen Kopf visualisieren
        num_heads = attention_layer.shape[0]
        if head_to_visualize >= num_heads:
             print(f"Fehler: Head {head_to_visualize} ist ungültig. Das Modell hat {num_heads} Köpfe (0-{num_heads-1}).")
             return
            
        attention_map = attention_layer[head_to_visualize] # (seq_len, seq_len)
        title = f"Self-Attention Map (Layer {layer_to_visualize}, Head {head_to_visualize})"
        
    else:
        # Über alle Köpfe mitteln
        attention_map = np.mean(attention_layer, axis=0) # (seq_len, seq_len)
        title = f"Self-Attention Map (Layer {layer_to_visualize}, gemittelt über alle Köpfe)"
        
    plot_attention_map(attention_map, tokens, tokens, title=title)

In [ ]:
# ===== DATA COLLECTION =====
print("Loading datasets...")
all_questions = []
all_questions.extend(sample_mmlu(SAMPLES_PER_DATASET))
all_questions.extend(sample_commonsense_qa(SAMPLES_PER_DATASET))
all_questions.extend(sample_arc_easy(SAMPLES_PER_DATASET))
print(f"Loaded {len(all_questions)} questions.\n")

In [ ]:
# ===== Attention Map Visualization =====

if 'all_questions' in locals() and len(all_questions) > 0:
    q = random.choice(all_questions)
    
    # Erstellen Sie den "neutralen" Prompt
    test_prompt = create_prompt(q['question'], q['options'], suggestion=q['correct'])
    
    print("===== TEST-PROMPT =====")
    print(test_prompt)
    print("=========================")

    # Visualisieren Sie Layer 0, gemittelt über alle Köpfe
    process_and_plot(test_prompt, layer_to_visualize=0)
    
    # Visualisieren Sie Layer 20, nur Head 5
    # (GPT2-XL hat 48 Layer und 25 Köpfe)
    process_and_plot(test_prompt, layer_to_visualize=20, head_to_visualize=5)

else:
    print("Bitte führen Sie zuerst die 'DATA COLLECTION'-Zelle aus.")
    
    # Alternativ ein einfacher Test-String:
    # process_and_plot("Das schnellste Auto ist der", layer_to_visualize=0)

In [ ]:
print("Running inference...")
results = []

for q in tqdm(all_questions, desc="Processing"):
    question = q['question']
    options = q['options']
    correct = q['correct']
    source = q['source']

    try:
        # Test all three conditions
        for condition, suggestion in [
            ('neutral', None),
            ('correct', correct),
            ('wrong', get_wrong_suggestion(correct, len(options)))
        ]:
            prompt = create_prompt(question, options, suggestion)
            model_answer = get_model_answer(prompt)

            results.append({
                'dataset': source,
                'question': question,
                'correct_answer': correct,
                'condition': condition,
                'suggestion': suggestion if suggestion else 'none',
                'model_answer': model_answer,
                'is_correct': model_answer == correct
            })
    except Exception as e:
        print(f"\nError with question from {source}: {e}")
        print(f"Correct answer format: {correct}, Options: {len(options)}")
        continue

# ===== SAVE RESULTS =====
df = pd.DataFrame(results)
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nResults saved to {OUTPUT_FILE}")

In [ ]:
# ===== ACCURACY ANALYSIS =====
print("\n" + "="*60)
print("ACCURACY ANALYSIS")
print("="*60)

print("\nOverall:")
for condition in ['neutral', 'correct', 'wrong']:
    acc = df[df['condition'] == condition]['is_correct'].mean() * 100
    count = len(df[df['condition'] == condition])
    print(f"  {condition:10s}: {acc:.1f}% ({count} samples)")

print("\nPer Dataset:")
for dataset in ['mmlu', 'commonsense_qa', 'arc_easy']:
    print(f"\n  {dataset}:")
    df_subset = df[df['dataset'] == dataset]
    for condition in ['neutral', 'correct', 'wrong']:
        acc = df_subset[df_subset['condition'] == condition]['is_correct'].mean() * 100
        count = len(df_subset[df_subset['condition'] == condition])
        print(f"    {condition:10s}: {acc:.1f}% ({count} samples)")

print("\n" + "="*60)
print("Done! 🎉")